# Week 4 — PhoBERT v2 + LLM Cascade


In [ ]:
# Cell 1
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}  VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Khong co GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 tokenizers==0.19.1 accelerate==0.30.1 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece faiss-cpu
!pip install -q openai google-generativeai
print('Dependencies installed')
print('If Kaggle still keeps old packages in memory, restart session and run all cells again.')

In [ ]:
# Cell 2
import os, sys

REPO_URL    = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for d in ['outputs/models', 'outputs/results', 'outputs/results/week4_augmented_v2',
          'outputs/results/week4_augmented_v2/models', 'outputs/eda']:
    os.makedirs(d, exist_ok=True)

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    sys.path.insert(0, p)

for f in ['data/train_preprocessed.csv', 'data/dev_preprocessed.csv',
          'data/test_preprocessed.csv', 'data/train_augmented_v2.csv']:
    print(f'  [{"OK" if os.path.exists(f) else "MISSING"}] {f}')

In [ ]:
# Cell 3
from kaggle_secrets import UserSecretsClient

secrets  = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')

API_KEY = OPENAI_API_KEY

In [ ]:
# Cell 4
import os

required_files = [
    'data/train_augmented_v2.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    'outputs/eda/encoder_config.json',
]

for f in required_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f} ({size:,} bytes)')

assert all(os.path.exists(f) for f in required_files), 'Missing files!'
print('\nAll files verified.')

In [ ]:
# Cell 5
import json

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
cw      = json.load(open('outputs/eda/class_weights.json'))

print('Encoder Config:')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

print('\nGlobal Class Weights:')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = ' <- clip 10.0' if float(w) > 10 else ''
    print(f"  {label_map.get(cls, cls):12s}: {float(w):.1f}x{note}")

In [ ]:
# Cell 5b
import inspect, glob

from transformers import AutoTokenizer
from train import train, load_class_weights
from model import ABSAPhoBERT
from step2_dataloader import create_dataloaders
from predict import load_best_model, predict_and_evaluate
from llm_client import LLMClient
from explainer import explain_predictions
from run_demo import format_output
from cascade_predictor import run_cascade_on_test, predict_single_phobert

print('Imports OK')
print()

for fn in [train, load_best_model, predict_and_evaluate, create_dataloaders, load_class_weights,
           run_cascade_on_test, predict_single_phobert]:
    params = list(inspect.signature(fn).parameters.keys())
    print(f'  {fn.__name__}: {params}')

print()
for f in ['data/train_augmented_v2.csv', 'data/dev_preprocessed.csv',
          'data/test_preprocessed.csv', 'outputs/eda/class_weights.json']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

missing = [f for f in ['data/train_augmented_v2.csv', 'data/dev_preprocessed.csv',
                        'data/test_preprocessed.csv', 'outputs/eda/class_weights.json']
           if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Missing: {missing}')

print('\nPre-flight OK')

In [ ]:
# Cell 6
import os, torch
from transformers import AutoTokenizer
from train import train, load_class_weights
from model import ABSAPhoBERT
from step2_dataloader import create_dataloaders
from predict import load_best_model, predict_and_evaluate
from utils.helpers import set_seed

torch.cuda.empty_cache()
set_seed(42)

device   = torch.device('cuda')
SAVE_DIR = 'outputs/results/week4_augmented_v2'
TRAIN_PATH = 'data/train_augmented_v2.csv'
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)

config = {
    'learning_rate':           3e-5,
    'warmup_ratio':            0.15,
    'batch_size':              16,
    'grad_accumulation_steps': 1,
    'max_epochs':              35,
    'early_stop_patience':     7,
    'dropout':                 0.2,
    'seed':                    42,
    'max_seq_len':             256,
    'weight_clip':             10.0,
    'encoder_option':          'cls_only',
    'lr_alpha':                0.1,
}

tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')

train_loader, dev_loader, test_loader = create_dataloaders(
    train_path=TRAIN_PATH,
    dev_path='data/dev_preprocessed.csv',
    test_path='data/test_preprocessed.csv',
    tokenizer=tokenizer,
    batch_size=config['batch_size'],
    max_len=config['max_seq_len'],
    num_workers=2,
    use_preprocessed=True,
)

model = ABSAPhoBERT(
    model_name='vinai/phobert-base-v2',
    encoder_option=config['encoder_option'],
    dropout=config['dropout'],
).to(device)

class_weights = load_class_weights(
    'outputs/eda/class_weights.json',
    weight_clip=config['weight_clip'],
    device=torch.device('cpu'),
)

history = train(
    model, train_loader, dev_loader, class_weights, device,
    config,
    save_dir=f'{SAVE_DIR}/models',
    results_dir=SAVE_DIR,
    use_amp=True,
)

model = load_best_model(f'{SAVE_DIR}/models/best_model.pt', model, device)

dev_metrics, _, _ = predict_and_evaluate(
    model, dev_loader, class_weights, device,
    split_name='dev',
    save_path=f'{SAVE_DIR}/week4_dev_metrics.json',
)
test_metrics, _, _ = predict_and_evaluate(
    model, test_loader, class_weights, device,
    split_name='test',
    save_path=f'{SAVE_DIR}/week4_test_metrics.json',
)

print(f"Dev  Combined F1: {dev_metrics['macro_combined_f1']:.4f}")
print(f"Test ACD F1:      {test_metrics['macro_acd_f1']:.4f}")
print(f"Test SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"Test Combined F1: {test_metrics['macro_combined_f1']:.4f}")

In [ ]:
# Cell 7
import json, os

w2_candidates = [
    'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json',
    'outputs/results/week2_results_VNcoreNLP/models_cls_only/week2_test_metrics.json',
    'outputs/results/week2_results_NO_ VNcoreNLP/models_cls_only/week2_test_metrics.json',
    'outputs/results_cls_only/week2_test_metrics.json',
    'outputs/results/week2_test_metrics.json',
]
w4_candidates = [
    'outputs/results/week4_augmented_v2/week4_test_metrics.json',
    'outputs/results/week4_augmented_v2/test_metrics.json',
]

w2_m = next((json.load(open(p)) for p in w2_candidates if os.path.exists(p)), None)
w4_m = next((json.load(open(p)) for p in w4_candidates if os.path.exists(p)), None)

print(f"{'Metric':<20} {'Baseline':>10} {'Aug-v2':>10} {'Delta':>8}")
print('-' * 52)
for key, label in [('macro_acd_f1', 'ACD F1'),
                   ('macro_spc_f1', 'SPC F1'),
                   ('macro_combined_f1', 'Combined F1')]:
    v2    = w2_m.get(key, 0) if w2_m else 0
    v4    = w4_m.get(key, 0) if w4_m else 0
    delta = v4 - v2
    print(f'{label:<20} {v2:>10.4f} {v4:>10.4f} {delta:>+8.4f}')

if not w2_m:
    print('\nBaseline metrics missing — check repo pull in Cell 2.')
if not w4_m:
    print('\nWeek 4 metrics missing — run Cell 6 first.')

---
## Phase C + Explainability

Cell 8 chạy cascade PhoBERT + LLM RAG cho WEAK_ASPECTS uncertain. Sau đó Cell 9-11 giữ luồng explainability.

In [ ]:
# Cell 8
import os, json, torch
import pandas as pd
from transformers import AutoTokenizer
from model import ABSAPhoBERT
from step4_eval import evaluate_predictions
from llm_client import LLMClient
from rag_retriever import ABSARetriever
from cascade_predictor import run_cascade_on_test
from utils.constants import ZERO_TRAIN_ASPECTS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = next(
    (c for c in [
        'outputs/results/week4_augmented_v2/models/best_model.pt',
        'outputs/results/week4_augmented/models/best_model.pt',
        'outputs/models_cls_only/best_model.pt',
        'outputs/models/best_model.pt',
    ] if os.path.exists(c)),
    None,
)
if not CHECKPOINT:
    raise FileNotFoundError('No checkpoint found. Run Cell 6 first.')
if not API_KEY:
    raise ValueError('Missing API_KEY for cascade eval.')

train_df = pd.read_csv('data/train_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')

ckpt = torch.load(CHECKPOINT, map_location=device)
encoder_option = ckpt.get('config', {}).get('encoder_option', 'cls_only')
model_cascade = ABSAPhoBERT(model_name='vinai/phobert-base-v2', encoder_option=encoder_option)
model_cascade.load_state_dict(ckpt['model_state_dict'])
model_cascade = model_cascade.to(device).eval()

retriever = ABSARetriever(cache_path='outputs/results/embeddings_cache.npy', use_faiss=True)
retriever.fit(train_df)
client = LLMClient(provider='openai', api_key=API_KEY)

y_true_c, y_pred_c, cascade_stats = run_cascade_on_test(
    test_df=test_df,
    train_df=train_df,
    model=model_cascade,
    tokenizer=tokenizer,
    device=device,
    retriever=retriever,
    llm_client=client,
    threshold=0.60,
    k=4,
    sleep_sec=1.0,
)

cascade_metrics = evaluate_predictions(
    y_true_c,
    y_pred_c,
    title='Week 4 v2 Cascade',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path='outputs/results/week4_augmented_v2/week4_cascade_test_metrics.json',
)

with open('outputs/results/week4_augmented_v2/week4_cascade_stats.json', 'w', encoding='utf-8') as f:
    json.dump(cascade_stats, f, ensure_ascii=False, indent=2)

print(json.dumps(cascade_stats, ensure_ascii=False, indent=2))
print(f"Cascade Combined F1: {cascade_metrics['macro_combined_f1']:.4f}")

In [ ]:
# Cell 9
import os, torch
import pandas as pd
from transformers import AutoTokenizer
from model import ABSAPhoBERT
from utils.constants import ASPECT_COLUMNS, IDX_TO_LABEL

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHECKPOINT = next(
    (c for c in [
        'outputs/results/week4_augmented_v2/models/best_model.pt',
        'outputs/results/week4_augmented/models/best_model.pt',
        'outputs/models_cls_only/best_model.pt',
        'outputs/models/best_model.pt',
    ] if os.path.exists(c)),
    None,
)
if not CHECKPOINT:
    raise FileNotFoundError('No checkpoint found. Run Cell 6 first.')

ckpt = torch.load(CHECKPOINT, map_location=device)
assert 'model_state_dict' in ckpt, f"Bad checkpoint keys: {list(ckpt.keys())}"

encoder_option = config['encoder_option'] if 'config' in dir() else ckpt.get('config', {}).get('encoder_option', 'cls_only')

model_inf = ABSAPhoBERT(model_name='vinai/phobert-base-v2', encoder_option=encoder_option)
model_inf.load_state_dict(ckpt['model_state_dict'])
model_inf = model_inf.to(device).eval()
print(f'Loaded {CHECKPOINT}  epoch={ckpt.get("epoch")}  combined_f1={ckpt.get("combined_f1", 0):.4f}')

tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')

test_df   = pd.read_csv('data/test_preprocessed.csv')
reviews   = test_df['Review'].tolist()
processed = test_df['processed_review'].tolist()

predictions_list = []
with torch.no_grad():
    for text in processed:
        inputs = tokenizer(text, max_length=256, padding='max_length',
                           truncation=True, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items() if k in ('input_ids', 'attention_mask')}
        preds  = model_inf(**inputs)['preds'][0].cpu().numpy()
        predictions_list.append({
            asp: IDX_TO_LABEL[int(preds[j])]
            for j, asp in enumerate(ASPECT_COLUMNS)
            if int(preds[j]) > 0
        })

print(f'Inference done: {len(reviews)} reviews, {sum(1 for p in predictions_list if p)} with aspects')
torch.cuda.empty_cache()

In [ ]:
# Cell 10
import time, json
from llm_client import LLMClient
from explainer import explain_predictions
from run_demo import format_output

client = LLMClient(provider='openai', api_key=API_KEY)

sorted_idx   = sorted(range(len(reviews)), key=lambda i: len(predictions_list[i]), reverse=True)
demo_indices = [i for i in sorted_idx if predictions_list[i]][:5]

all_results = []
for rank, idx in enumerate(demo_indices):
    review       = reviews[idx]
    preds        = predictions_list[idx]
    print(f'[{rank+1}/5] Review {idx} ({len(preds)} aspects)...')
    explanations = explain_predictions(review, preds, client)
    result       = {'review': review, 'predictions': preds, 'explanations': explanations}
    all_results.append(result)
    print(format_output(result))
    if rank < len(demo_indices) - 1:
        time.sleep(1.0)

In [ ]:
# Cell 11
import shutil, json, os

out_path = 'outputs/results/week4_explainer_samples.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'Saved {len(all_results)} explained reviews -> {out_path}')

total_asp  = sum(len(r['predictions']) for r in all_results)
total_expl = sum(len(r['explanations']) for r in all_results)
print(f'Aspects detected: {total_asp}, explained: {total_expl}')

cascade_path = 'outputs/results/week4_augmented_v2/week4_cascade_test_metrics.json'
if os.path.exists(cascade_path):
    with open(cascade_path, 'r', encoding='utf-8') as f:
        cascade_m = json.load(f)
    print(f"Cascade Combined F1  : {cascade_m.get('macro_combined_f1', 0):.4f}")

shutil.make_archive('/kaggle/working/week4_results', 'zip', 'outputs')
print('Zip: /kaggle/working/week4_results.zip  (Kaggle Output panel -> Download)')

if w2_m and w4_m:
    d = w4_m.get('macro_combined_f1', 0) - w2_m.get('macro_combined_f1', 0)
    print(f"\nBaseline Combined F1 : {w2_m.get('macro_combined_f1', 0):.4f}")
    print(f"Aug-v2 Combined F1   : {w4_m.get('macro_combined_f1', 0):.4f}  ({d:+.4f})")

---
## Phase Final — PhoBERT v1 Retrain + LLM Cascade

Các cell bên dưới giữ nguyên notebook cũ và thêm luồng mới cho ablation `vinai/phobert-base` vs `vinai/phobert-base-v2`, sau đó chạy cascade trên `WEAK_ASPECTS` bằng RAG k=4.

In [ ]:
# Cell F1 — Setup PhoBERT v2 cascade on Kaggle
import os, sys, json, torch
import pandas as pd
from transformers import AutoTokenizer

REPO_ROOT = os.getcwd()
for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    full = os.path.join(REPO_ROOT, p)
    if full not in sys.path:
        sys.path.insert(0, full)

from utils.constants import PHOBERT_V2, TRAIN_CONFIG, WEAK_ASPECTS, ZERO_TRAIN_ASPECTS
from utils.helpers import set_seed, save_json
from step4_eval import evaluate_predictions
from model import ABSAPhoBERT
from predict import load_best_model
from rag_retriever import ABSARetriever
from llm_client import LLMClient
from cascade_predictor import run_cascade_on_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(TRAIN_CONFIG['seed'])

BEST_MODEL_NAME = PHOBERT_V2
BEST_ENCODER = 'cls_only'
BEST_CHECKPOINT = 'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt'
BEST_RESULTS_DIR = 'outputs/results/week2_results_VNcoreNLP'
V2_TEST_METRICS = 'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json'
v2_metrics = json.load(open(V2_TEST_METRICS, encoding='utf-8'))

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')
print(f'Base model: {BEST_MODEL_NAME}')
print(f'Checkpoint: {BEST_CHECKPOINT}')
print(f'Baseline Combined F1: {v2_metrics["macro_combined_f1"]:.4f}')
print(f'WEAK_ASPECTS ({len(WEAK_ASPECTS)}): {WEAK_ASPECTS}')


In [ ]:
# Cell F2 — Load PhoBERT v2 and smoke test cascade on 10 reviews
BEST_TOKENIZER = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
BEST_MODEL = ABSAPhoBERT(
    model_name=BEST_MODEL_NAME,
    dropout=TRAIN_CONFIG['dropout'],
    encoder_option=BEST_ENCODER,
).to(device)
BEST_MODEL = load_best_model(BEST_CHECKPOINT, BEST_MODEL, device)

train_df = pd.read_csv('data/train_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')
retriever = ABSARetriever(cache_path='outputs/results/embeddings_cache.npy', use_faiss=True)
retriever.fit(train_df)

api_key = os.environ.get('OPENAI_API_KEY', API_KEY if 'API_KEY' in globals() else '')
if not api_key:
    raise ValueError('Missing OPENAI_API_KEY / API_KEY for cascade.')
llm_client = LLMClient(provider='openai', api_key=api_key)

SMOKE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4_smoke')
os.makedirs(SMOKE_DIR, exist_ok=True)

y_true_smoke, y_pred_smoke, smoke_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=0.60,
    k=4,
    sleep_sec=0.0,
    max_samples=10,
    return_records=True,
)

smoke_metrics = evaluate_predictions(
    y_true_smoke,
    y_pred_smoke,
    title='Cascade Smoke Test (10 reviews) — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(SMOKE_DIR, 'cascade_smoke_metrics.json'),
)
save_json(smoke_stats, os.path.join(SMOKE_DIR, 'cascade_smoke_stats.json'))

print(f'Base model for cascade: {BEST_MODEL_NAME}')
print(json.dumps(smoke_stats, ensure_ascii=False, indent=2))
print(f"Smoke Combined F1: {smoke_metrics['macro_combined_f1']:.4f}")


In [ ]:
# Cell F3 — Full cascade on test set using PhoBERT v2 + WEAK_ASPECTS + RAG k=4
CASCADE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4')
os.makedirs(CASCADE_DIR, exist_ok=True)

y_true_cascade, y_pred_cascade, cascade_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=0.60,
    k=4,
    sleep_sec=1.0,
    return_records=True,
)

cascade_metrics = evaluate_predictions(
    y_true_cascade,
    y_pred_cascade,
    title='Cascade Test — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(CASCADE_DIR, 'cascade_test_metrics.json'),
)
save_json(cascade_stats, os.path.join(CASCADE_DIR, 'cascade_stats.json'))

comparison = {
    'baseline_v2_cls_only': {
        'model': BEST_MODEL_NAME,
        'combined_f1': v2_metrics['macro_combined_f1'],
        'acd_f1': v2_metrics['macro_acd_f1'],
        'spc_f1': v2_metrics['macro_spc_f1'],
    },
    'cascade_v2_rag_k4': {
        'model': BEST_MODEL_NAME,
        'combined_f1': cascade_metrics['macro_combined_f1'],
        'acd_f1': cascade_metrics['macro_acd_f1'],
        'spc_f1': cascade_metrics['macro_spc_f1'],
        'trigger_rate': cascade_stats['trigger_rate'],
        'total_overrides': cascade_stats['total_overrides'],
    },
}
save_json(comparison, os.path.join(CASCADE_DIR, 'baseline_vs_cascade_comparison.json'))

print(f'WEAK_ASPECTS used: {WEAK_ASPECTS}')
print(json.dumps(cascade_stats, ensure_ascii=False, indent=2))
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f"Cascade Combined F1: {cascade_metrics['macro_combined_f1']:.4f}")
